# LLM-Proto: Add & Train a Mixture-of-Experts (MoE) Layer

Take a **pre-trained `TransformerLM` checkpoint** (loaded locally or from Google Drive), graft sparse MoE expert layers on top of it, then fine-tune **only the new expert parameters** — the base model stays frozen.

**Why MoE?**  
A Mixture-of-Experts FFN routes each token to `top_k` out of `n_experts` sub-networks.  
Total parameter count grows with `n_experts`, but compute per token stays constant — the same cost as one FFN.  
This is how Mixtral, Gemini, and GPT-4 scale: more parameters without proportionally more FLOPS.

**Workflow:**
1. Install dependencies
2. Settings — base model, expert config, fine-tuning data
3. Google Drive setup — mount Drive and verify base checkpoint availability
3b. Prepare domain-specific fine-tuning data from HuggingFace
4. Load base model from checkpoint (local or Google Drive)
5. Sample output *before* expert fine-tuning (baseline)
6. Define `SparseMoE` (router + expert FFNs)
7. Graft experts into selected transformer blocks
8. Freeze base weights — only experts are trainable
9. Prepare fine-tuning dataloader
10. Expert fine-tuning loop (LM loss + load-balance auxiliary loss)
11. Sample output *after* expert fine-tuning
12. Save expert checkpoint & upload to Google Drive
13. Reload expert checkpoint
14. Expert load distribution

---

## 1. Install Dependencies

In [ ]:
# ── Cell 1: Install Dependencies ──
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "torch>=2.1.0", "tokenizers", "datasets", "wandb", "safetensors",
    "matplotlib", "seaborn", "numpy", "pyyaml", "tqdm",
    "google-api-python-client>=2.100.0", "google-auth>=2.23.0",
]:
    install(pkg)

print("All dependencies installed!")

## 2. Imports & Settings

| Setting | What it controls |
|---------|------------------|
| `BASE_MODEL_SIZE` | Architecture preset that must match your checkpoint (`"tiny"`, `"small"` …) |
| `CKPT_DIR` / `CKPT_FILE` | Location of the pre-trained checkpoint to load |
| `EXPERT_LAYERS` | Which transformer block indices get an MoE FFN — `None` = all layers |
| `N_EXPERTS` | Total number of expert FFNs per MoE layer (e.g. 4) |
| `TOP_K` | How many experts each token activates (e.g. 2 out of 4) |
| `WARM_START_EXPERTS` | If `True`, copy base FFN weights into every expert before fine-tuning |
| `EXPERT_CKPT_DIR` | Where to save the expert-augmented checkpoint |
| `FINETUNE_DATA_DIR` | Local directory with tokenized binary shards (auto-updated by cell 2c) |
| `FINETUNE_HF_DATASETS` | HuggingFace dataset IDs to download and tokenize for domain fine-tuning |
| `N_STEPS` | Number of fine-tuning steps |
| `AUX_LOSS_COEFF` | Weight of the load-balance auxiliary loss (keeps experts equally loaded) |

In [ ]:
# ── Cell 2: Imports & Settings ──

# ── MUST be set before CUDA context is created (i.e. before any torch.cuda calls) ──
# Tells the allocator to use expandable memory segments, which drastically reduces
# memory fragmentation when many tensors of varying sizes are allocated and freed
# (exactly what happens with gradient checkpointing + MoE routing).
# Requires a fresh kernel — restart the runtime after changing this.
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import sys, math, time, copy, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from contextlib import nullcontext

# Make sure we can import src/
if os.path.basename(os.getcwd()) != "LLM-Proto":
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "https://github.com/VlSePr/LLM-Proto"], check=True)
        os.chdir("LLM-Proto")

from src.config import ModelConfig, TrainConfig, get_model_config, load_model_config
from src.model import TransformerLM, FeedForward, TransformerBlock, RMSNorm
from src.tokenizer import LLMTokenizer
from src.data import create_dataloader
from src.utils import (
    detect_environment, get_device, get_dtype, set_seed,
    get_lr, save_checkpoint, load_checkpoint, has_checkpoint,
)
from src.generate import generate_text

# ─────────────────────────────────────────
# Settings  (edit these)
# ─────────────────────────────────────────

# Base model
BASE_MODEL_SIZE     = "large"         # Must match the checkpoint architecture
TOKENIZER_PATH      = "tokenizer_data"
CKPT_DIR            = "checkpoints"  # Folder with the base checkpoint
CKPT_FILE           = "latest.pt"    # "latest.pt", "best.pt", "step_5000.pt" …

# Google Drive — base model (where to pull the checkpoint to extend)
ENABLE_GDRIVE       = True
GDRIVE_FOLDER_ID    = "LLM"          # Drive folder containing the base checkpoint
GDRIVE_CREDENTIALS  = ""             # Service-account JSON path (local only)

# Google Drive — expert model (where to push the trained expert checkpoint)
EXPERT_GDRIVE_FOLDER_ID = "LLM"     # Can be the same folder or a dedicated one

# Expert architecture
# ── Incremental growth: start with 1 expert, expand one at a time ──
# Round 1: N_EXPERTS=1, TOP_K=1  → run cells 5–11 to train and save
# Round 2: run cell 12b to expand → N_EXPERTS becomes 2, TOP_K stays 1 or bump to 2
# Round N: repeat — each round trains only the newest expert; older experts stay frozen
#
# ── Expert layer placement ──
# With dim=2048 and ffn_dim=5632, each expert FFN is 34.6M params.
# The number of trainable params = len(EXPERT_LAYERS) × N_EXPERTS × 34.6M
#
#   EXPERT_LAYERS = list(range(32))     → all 32 layers → 1.1B trainable (risky on VRAM)
#   EXPERT_LAYERS = list(range(24, 32)) → last 8 layers → 277M trainable (recommended)
#   EXPERT_LAYERS = list(range(16, 32)) → last 16 layers → 554M trainable (moderate)
#
# Upper layers specialize in semantics / output distributions — ideal for domain tuning.
# Lower layers encode low-level syntax and are better left untouched.
EXPERT_LAYERS       = list(range(24, 32))  # last 8 layers → 277M trainable
N_EXPERTS           = 1              # Start with 1; cell 12b increments this each round
TOP_K               = 1              # Experts activated per token (must be ≤ N_EXPERTS)
WARM_START_EXPERTS  = True           # Copy base FFN weights into each expert (recommended)

# Fine-tuning
FINETUNE_DATA_DIR   = "data/custom/bins"  # Tokenized shards dir (auto-updated by cell 2c)
FINETUNE_HF_DATASETS = [              # HuggingFace datasets for Dark Souls domain tuning
        "SantaBot/dark-souls-lore",       # 36 rows -- Dark Souls item/lore descriptions
        "antoshka1608/dark_souls_remastered",  # 534 rows -- DS enemy/item descriptions
        "AINovice2005/EldenRing_Small",   # 49k rows -- large Elden Ring lore corpus
        "ArenaRune/EldenRingQA",          # 11.7k rows -- Elden Ring QA pairs (lore + mechanics)
    ]                                     # Set to [] to skip HF download
EXPERT_CKPT_DIR     = "checkpoints/expert"  # Where to save the expert checkpoint

# ── Sequence length for fine-tuning ──
# Fine-tuning on domain data doesn't need the full 4096-token context the base model supports.
# With gradient checkpointing, activation memory scales as O(n_layers × B × T × dim).
# Halving T from 4096 → 2048 saves ~32 GB of activation checkpoints on the large model.
# The checkpoint-recomputation intermediates (MoE FFN pass) also scale with B×T.
#
# Memory budget (large model, 24 MoE layers → last 8):
#   T=4096, B=16 → OOM at 94 GB  (each checkpoint tensor = 268 MB × 32 = 8.5 GB alone)
#   T=1024, B=4  → ≈18 GB total  (each checkpoint tensor = 17 MB × 32 = 530 MB)
FINETUNE_SEQ_LEN    = 1024           # Tokens per sample during fine-tuning (≤ max_seq_len)
N_STEPS             = 2000           # Fine-tuning steps
BATCH_SIZE          = 4              # Micro-batch size (reduce further if still OOM)
GRAD_ACCUM          = 16             # Effective batch = BATCH_SIZE × GRAD_ACCUM = 64
PEAK_LR             = 3e-4
MIN_LR              = 1e-5
WARMUP_STEPS        = 100
AUX_LOSS_COEFF      = 1e-2           # Load-balance auxiliary loss coefficient
SEED                = 42

# ─────────────────────────────────────────

device = get_device()
dtype  = get_dtype()
print(f"Environment : {detect_environment()}")
print(f"Device      : {device}")
print(f"Dtype       : {dtype}")
print(f"ALLOC_CONF  : {os.environ.get('PYTORCH_ALLOC_CONF', '(not set)')}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name()}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

set_seed(SEED)

## 2b. Google Drive Setup

Mounts Google Drive on Colab (or verifies API credentials in other environments).  
Lists available base checkpoints so you can confirm `CKPT_FILE` exists in `GDRIVE_FOLDER_ID` before the load step.

> **Skip** if `ENABLE_GDRIVE = False` — the cell is a no-op in that case.

In [ ]:
# ── Cell 2b: Google Drive Setup ──
# Mounts Google Drive on Colab (or verifies API credentials locally).
# Lists available base checkpoints so you can confirm CKPT_FILE exists before loading.

if ENABLE_GDRIVE:
    is_colab = "google.colab" in sys.modules

    if is_colab:
        from google.colab import drive
        MOUNT_POINT = "/content/drive"
        if not os.path.ismount(MOUNT_POINT):
            drive.mount(MOUNT_POINT)
            print(f"Google Drive mounted at {MOUNT_POINT}")
        else:
            print(f"Google Drive already mounted at {MOUNT_POINT}")

        # Show available base checkpoints
        if GDRIVE_FOLDER_ID:
            ckpt_drive_path = os.path.join(MOUNT_POINT, "MyDrive", GDRIVE_FOLDER_ID)
            os.makedirs(ckpt_drive_path, exist_ok=True)
            pts = sorted(f for f in os.listdir(ckpt_drive_path) if f.endswith(".pt"))
            print(f"\nBase checkpoint folder : {ckpt_drive_path}")
            print(f"Available checkpoints  : {pts if pts else '(none)'}")

        # Show expert output folder
        if EXPERT_GDRIVE_FOLDER_ID and EXPERT_GDRIVE_FOLDER_ID != GDRIVE_FOLDER_ID:
            expert_drive_path = os.path.join(MOUNT_POINT, "MyDrive", EXPERT_GDRIVE_FOLDER_ID)
            os.makedirs(expert_drive_path, exist_ok=True)
            print(f"Expert upload folder   : {expert_drive_path}")
    else:
        if GDRIVE_CREDENTIALS and os.path.isfile(GDRIVE_CREDENTIALS):
            print(f"Google Drive API credentials : {GDRIVE_CREDENTIALS}")
        else:
            print("Google Drive API mode (Application Default Credentials)")

        try:
            from src.gdrive import list_remote_checkpoints
            remote = list_remote_checkpoints(GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
            print(f"\nBase checkpoints on Google Drive ({len(remote)}):")
            for f in remote:
                print(f"  {f['name']:25s}  {f.get('createdTime', '')}")
        except Exception as e:
            print(f"Could not list Drive checkpoints: {e}")

    print("\nGoogle Drive integration: ENABLED")
else:
    print("Google Drive disabled — checkpoints will be loaded/saved locally only.")

## 2c. Prepare Domain-Specific Fine-Tuning Data

Downloads the **Dark Souls / Elden Ring** lore corpus from HuggingFace and tokenizes it  
into binary shards in `data/custom/bins/` that `create_dataloader` will read.

Datasets blended (edit `FINETUNE_HF_DATASETS` in Settings to customise):
* `SantaBot/dark-souls-lore` -- 36 rows, authentic Dark Souls item/lore descriptions  
* `antoshka1608/dark_souls_remastered` -- 534 rows, DS enemy/item descriptions  
* `AINovice2005/EldenRing_Small` -- **49 k rows**, large Elden Ring lore corpus  
* `ArenaRune/EldenRingQA` -- 11.7 k structured QA pairs (lore + mechanics)

> **Skip** (set `FINETUNE_HF_DATASETS = []`) if you already have tokenized data in `FINETUNE_DATA_DIR`.

In [ ]:
# Cell 2c: Prepare Domain-Specific Fine-Tuning Data
# Downloads Dark Souls / Elden Ring datasets from HuggingFace, writes a JSONL corpus,
# then tokenizes and packs into binary shards ready for create_dataloader.
import json as _json

if not FINETUNE_HF_DATASETS:
    print("FINETUNE_HF_DATASETS is empty -- using FINETUNE_DATA_DIR as-is:", FINETUNE_DATA_DIR)
else:
    try:
        from datasets import load_dataset as _hf_load
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset as _hf_load

    # Per-dataset text extractor -- QA datasets get question+answer concatenated
    _FIELD_MAP = {
        "AINovice2005/EldenRing_Small": lambda r: r.get("text", ""),
        "ArenaRune/EldenRingQA": lambda r: "\n".join(filter(None, [r.get("instruction", ""), r.get("output", "")])),
    }

    def _extract_text(row):
        for k in ("text", "description", "lore", "content", "body"):
            v = row.get(k, "")
            if isinstance(v, str) and len(v) > 20:
                return v
        return " ".join(str(v) for v in row.values() if isinstance(v, str))

    # Download & collect
    all_texts = []
    for ds_id in FINETUNE_HF_DATASETS:
        print(f"\nLoading {ds_id} ...")
        try:
            ds = _hf_load(ds_id, split="train", trust_remote_code=False)
            extractor = _FIELD_MAP.get(ds_id, _extract_text)
            texts = [extractor(r) for r in ds]
            texts = [t.strip() for t in texts if t and len(t.strip()) > 20]
            all_texts.extend(texts)
            print(f"  collected {len(texts):,} samples  (total: {len(all_texts):,})")
        except Exception as e:
            print(f"  FAILED {ds_id}: {e}")

    print(f"\nTotal samples collected: {len(all_texts):,}")

    # Write JSONL corpus
    _jsonl_dir = os.path.join("data", "custom", "jsonl")
    os.makedirs(_jsonl_dir, exist_ok=True)
    _corpus_path = os.path.join(_jsonl_dir, "dark_souls_corpus.jsonl")
    with open(_corpus_path, "w", encoding="utf-8") as _fh:
        for t in all_texts:
            _fh.write(_json.dumps({"text": t}, ensure_ascii=False) + "\n")
    print(f"Wrote {len(all_texts):,} entries -> {_corpus_path}")

    # Tokenize -> binary shards
    from src.data import tokenize_and_save
    _bins_dir = os.path.join("data", "custom", "bins")
    os.makedirs(_bins_dir, exist_ok=True)
    tokenize_and_save(
        tokenizer_path=TOKENIZER_PATH,
        output_dir=_bins_dir,
        sources=[{"type": "jsonl", "path": _corpus_path, "text_field": "text"}],
        shard_size=10_000_000,
        val_every=50,
    )
    # Update FINETUNE_DATA_DIR so the training loop uses this corpus
    FINETUNE_DATA_DIR = _bins_dir
    print(f"\nFINETUNE_DATA_DIR updated -> {FINETUNE_DATA_DIR}")

## 3. Load Base Model from Checkpoint

Builds the `TransformerLM` architecture and loads the pre-trained weights.  
The tokenizer path is read from checkpoint metadata so it always matches what was used during training.

> The base model is loaded in **eval mode** at first so we can capture its generation for comparison later.  
> After grafting experts it will be switched to train mode.

In [ ]:
# ── Cell 3: Load Base Model from Checkpoint ──

# Resolve model config
if os.path.isfile(BASE_MODEL_SIZE):
    model_config = load_model_config(BASE_MODEL_SIZE)
    print(f"Loaded model config from file: {BASE_MODEL_SIZE}")
else:
    model_config = get_model_config(BASE_MODEL_SIZE)
    print(f"Using preset model config: {BASE_MODEL_SIZE}")

# Build model skeleton
model = TransformerLM(model_config).to(device)
print(model.summary())

# Load checkpoint weights
gdrive_kw = {}
if ENABLE_GDRIVE and GDRIVE_FOLDER_ID:
    gdrive_kw = dict(gdrive_folder_id=GDRIVE_FOLDER_ID,
                     gdrive_credentials_path=GDRIVE_CREDENTIALS)

resume_name = CKPT_FILE.removesuffix(".pt")
ckpt = load_checkpoint(CKPT_DIR, resume_name, model, device=device, **gdrive_kw)
print(f"Checkpoint loaded: step {ckpt['step']}, loss {ckpt['loss']:.4f}")

# Tokenizer
tok_path = ckpt.get("train_config", {}).get("tokenizer_path", TOKENIZER_PATH)
tokenizer = LLMTokenizer(tok_path)
print(f"Tokenizer loaded (vocab_size={tokenizer.vocab_size})")

model.eval()
print("\nBase model ready.")

## 3b. Sample Output — Before Expert Fine-tuning

Generate a few text samples from the **base model** before MoE layers are added.  
These become the baseline — save or screenshot them to compare with the post-training output in step 11.

In [ ]:
# ── Cell 3b: Sample Output — Before Expert Training ──
# Baseline generation from the frozen base model before any MoE layers are added.
# Keep the same prompts as the post-training cell so results are directly comparable.

model.eval()

SAMPLE_PROMPTS = [
    "The ancient city of",
    "In the beginning",
    "Science has discovered that",
    "We didn`t start the fire, it was always burning"
]
GEN_MAX_TOKENS  = 80
GEN_TEMPERATURE = 0.8
GEN_TOP_K       = 40
GEN_TOP_P       = 0.9

print("=" * 60)
print("Base model — BEFORE expert fine-tuning")
print("=" * 60)
for prompt in SAMPLE_PROMPTS:
    out = generate_text(model, tokenizer, prompt,
                        GEN_MAX_TOKENS, GEN_TEMPERATURE, GEN_TOP_K, GEN_TOP_P, device)
    print(f"\nPrompt : {prompt}")
    print(f"Output : {out[:250]}")
print("=" * 60)

## 4. Define `SparseMoE` — Mixture-of-Experts FFN

The `SparseMoE` module is a **drop-in replacement** for the `FeedForward` block in each `TransformerBlock`.  
It adds `n_experts` independent SwiGLU FFNs and a lightweight **router** that, for each token, selects the `top_k` most relevant experts.

**Architecture:**
```
x (B, T, dim)
  └─► Router (linear dim→n_experts → softmax → top-k)   # O(dim × n_experts) params
  └─► Expert[i] (SwiGLU FFN, same as base FeedForward)  # top_k experts run per token
  └─► weighted sum of top-k expert outputs
```

**Load-balance auxiliary loss:**  
Without a regularization term the router collapses — it learns to always send everything to one expert.  
The Mixtral/Switch-Transformer recipe is:  
$$\mathcal{L}_{\text{aux}} = n_{\text{experts}} \cdot \sum_{i=1}^{n_{\text{experts}}} f_i \cdot P_i$$
where $f_i$ = fraction of tokens routed to expert $i$ (hard, non-differentiable),  
and $P_i$ = mean router probability for expert $i$ over the batch (soft, differentiable).  
The product $f_i \cdot P_i$ encourages uniform load while remaining trainable.

In [ ]:
# ── Cell 4: Define SparseMoE ──

class ExpertFFN(nn.Module):
    """
    A single SwiGLU expert — identical architecture to the base FeedForward.
    Having separate weight matrices per expert lets each one specialize.
    """
    def __init__(self, dim: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(dim, ffn_dim, bias=False)
        self.w_up   = nn.Linear(dim, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class SparseMoE(nn.Module):
    """
    Sparse Mixture-of-Experts FFN layer.

    Drop-in replacement for FeedForward in TransformerBlock.
    Routes each token independently to top_k out of n_experts expert FFNs.
    Returns the weighted sum of activated expert outputs.

    Also computes a load-balance auxiliary loss (stored in self.aux_loss).
    The training loop must add  `aux_loss_coeff * moe_layer.aux_loss`  to the
    total loss to encourage uniform expert utilization.
    """

    def __init__(self, dim: int, ffn_dim: int, n_experts: int, top_k: int,
                 dropout: float = 0.0):
        super().__init__()
        assert top_k <= n_experts, "top_k must be ≤ n_experts"

        self.n_experts = n_experts
        self.top_k     = top_k

        # Router: maps each token's hidden state to a distribution over experts.
        # Deliberately lightweight (no bias, no hidden layer) to keep overhead low.
        self.router = nn.Linear(dim, n_experts, bias=False)

        # Each expert is a full SwiGLU FFN
        self.experts = nn.ModuleList([
            ExpertFFN(dim, ffn_dim, dropout) for _ in range(n_experts)
        ])

        # Set by forward(); training loop reads this
        self.aux_loss: torch.Tensor = torch.tensor(0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        # Flatten batch and sequence dims so every token is treated independently
        tokens = x.view(-1, D)           # (B*T, dim)
        N = tokens.shape[0]              # total number of tokens in this batch

        # ── Router ──
        # Compute router logits and probabilities
        router_logits = self.router(tokens)            # (N, n_experts)
        router_probs  = F.softmax(router_logits, dim=-1)  # (N, n_experts)

        # Select top-k experts for each token
        topk_probs, topk_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # Re-normalize selected probabilities so they sum to 1 per token
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)  # (N, top_k)

        # ── Load-balance auxiliary loss ──
        # f_i  = fraction of tokens dispatched to expert i (non-differentiable)
        # P_i  = mean router probability for expert i over the batch (differentiable)
        # L_aux = n_experts * sum(f_i * P_i)  — low when experts are equally loaded
        # Only computed during training. At inference self.training is False.
        if self.training:
            with torch.no_grad():
                # one-hot dispatch mask: shape (N, n_experts), True where token was routed
                dispatch_mask = torch.zeros(N, self.n_experts, device=x.device)
                dispatch_mask.scatter_(1, topk_indices, 1.0)
                # f_i: fraction of tokens that chose expert i among top-k
                f = dispatch_mask.mean(dim=0)           # (n_experts,)
            # P_i: mean *soft* probability for expert i (differentiable)
            P = router_probs.mean(dim=0)                # (n_experts,)
            self.aux_loss = self.n_experts * (f * P).sum()
        else:
            self.aux_loss = torch.tensor(0.0, device=x.device)

        # ── Expert computation ──
        # output accumulator (zeros — will be filled by routing)
        output = torch.zeros_like(tokens)  # (N, dim)

        # For each expert, collect all tokens routed to it, run them through
        # the expert FFN in a single batched call, then scatter back.
        for expert_idx, expert in enumerate(self.experts):
            # Which token-slots (across ALL top-k positions) chose this expert?
            # topk_indices: (N, top_k) — positions where value == expert_idx
            token_mask = (topk_indices == expert_idx)  # (N, top_k) bool
            any_routed = token_mask.any(dim=1)         # (N,) — tokens that use this expert

            if not any_routed.any():
                continue  # No tokens routed here this step — skip entirely

            # Gather the tokens this expert handles
            expert_input  = tokens[any_routed]          # (n_routed, dim)
            expert_output = expert(expert_input)        # (n_routed, dim)

            # Gather the corresponding routing weights
            # For each routed token, pick the weight of this specific expert
            weights = topk_probs[any_routed]                    # (n_routed, top_k)
            slot_mask = token_mask[any_routed]                  # (n_routed, top_k)
            # Each token has exactly one weight for this expert across its top_k slots
            expert_weights = weights[slot_mask].unsqueeze(-1)   # (n_routed, 1)

            # Accumulate weighted expert output back into the output tensor
            output[any_routed] += expert_weights * expert_output

        return output.view(B, T, D)


print("SparseMoE and ExpertFFN classes defined.")

# Quick sanity check
_dim, _ffn_dim = model_config.dim, model_config.ffn_dim
_moe = SparseMoE(_dim, _ffn_dim, n_experts=N_EXPERTS, top_k=TOP_K).to(device)
_dummy = torch.randn(2, 16, _dim, device=device)
_moe.train()
_out = _moe(_dummy)
assert _out.shape == _dummy.shape, "Shape mismatch!"
print(f"Self-test passed — SparseMoE output: {_out.shape}, aux_loss={_moe.aux_loss.item():.4f}")
del _moe, _dummy, _out

## 5. Graft Experts into the Model

Replaces the `ffn` attribute of each targeted `TransformerBlock` with a `SparseMoE` module.  
The rest of the block (RMSNorm, Attention, residual connections) is left unchanged and shared.

**Warm-start vs. cold-start:**  
- **Warm-start** (`WARM_START_EXPERTS = True`): Copy the base FFN weights into every expert.  
  All experts start identical to the original FFN — they diverge during fine-tuning.  
  This is a better starting point when fine-tuning data is limited.  
- **Cold-start**: Experts are randomly initialized — useful if you want complete specialization
  and have enough data.

In [ ]:
# ── Cell 5: Graft SparseMoE into Selected Transformer Blocks ──

# Determine which layer indices to upgrade
target_layer_indices = EXPERT_LAYERS if EXPERT_LAYERS is not None else list(range(model_config.n_layers))
print(f"Grafting MoE into layers: {target_layer_indices}")
print(f"  n_experts={N_EXPERTS}, top_k={TOP_K}, warm_start={WARM_START_EXPERTS}")

for idx in target_layer_indices:
    block: TransformerBlock = model.layers[idx]
    base_ffn: FeedForward = block.ffn

    # Build MoE layer with same dims as the original FFN
    moe = SparseMoE(
        dim=model_config.dim,
        ffn_dim=model_config.ffn_dim,
        n_experts=N_EXPERTS,
        top_k=TOP_K,
        dropout=model_config.dropout,
    ).to(device)

    if WARM_START_EXPERTS:
        # Copy the base FFN weights into every expert so training starts from
        # a known-good point rather than random noise.
        with torch.no_grad():
            for expert in moe.experts:
                expert.w_gate.weight.copy_(base_ffn.w_gate.weight)
                expert.w_up.weight.copy_(base_ffn.w_up.weight)
                expert.w_down.weight.copy_(base_ffn.w_down.weight)

    # Replace FFN in-place
    block.ffn = moe

print("\nModel after expert grafting:")
print(model.summary())

# Verify MoE layers are present
moe_layers = [(i, layer.ffn) for i, layer in enumerate(model.layers)
              if isinstance(layer.ffn, SparseMoE)]
print(f"\nMoE layers: {[i for i, _ in moe_layers]}")

## 6. Freeze Base Weights — Only Train the Experts

All original model parameters are frozen (requires_grad = False).  
Only the newly added `SparseMoE` parameters receive gradients:
- `router` linear (tiny — `dim × n_experts` per layer)
- `experts[*].w_gate`, `w_up`, `w_down` (the bulk of new parameters)

In [ ]:
# ── Cell 6: Freeze Base Weights ──

# Step 1: freeze everything
for param in model.parameters():
    param.requires_grad_(False)

# Step 2: unfreeze only MoE parameters
moe_param_count = 0
for layer in model.layers:
    if isinstance(layer.ffn, SparseMoE):
        for param in layer.ffn.parameters():
            param.requires_grad_(True)
            moe_param_count += param.numel()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print("Parameter budget")
print(f"  Total              : {total_params:>12,}  ({total_params / 1e6:.1f}M)")
print(f"  Frozen (base)      : {frozen_params:>12,}  ({frozen_params / 1e6:.1f}M)")
print(f"  Trainable (experts): {trainable_params:>12,}  ({trainable_params / 1e6:.1f}M)")
print(f"  Trainable %        : {100 * trainable_params / total_params:.1f}%")

## 7. Prepare Fine-tuning Dataloader

Reuses the same `create_dataloader` from the base training pipeline.  
Point `FINETUNE_DATA_DIR` at a domain-specific dataset to specialize the experts.

In [ ]:
# ── Cell 7: Prepare Fine-tuning Data ──

train_loader = create_dataloader(
    FINETUNE_DATA_DIR,
    FINETUNE_SEQ_LEN,   # shorter than max_seq_len — reduces activation checkpoint size
    BATCH_SIZE,
    split="train",
    num_workers=2,
)
val_loader = create_dataloader(
    FINETUNE_DATA_DIR,
    FINETUNE_SEQ_LEN,
    BATCH_SIZE,
    split="val",
    num_workers=2,
    shuffle=False,
)

tokens_per_step = BATCH_SIZE * FINETUNE_SEQ_LEN * GRAD_ACCUM
print(f"Train loader: {len(train_loader)} batches")
print(f"Val   loader: {len(val_loader)} batches")
print(f"Seq len     : {FINETUNE_SEQ_LEN}  (model max: {model_config.max_seq_len})")
print(f"Tokens/step : {tokens_per_step:,}  (batch {BATCH_SIZE} × seq {FINETUNE_SEQ_LEN} × accum {GRAD_ACCUM})")

## 8. Expert Fine-tuning Loop

Standard gradient-accumulation loop with two loss terms:

| Loss term | What it trains | Why |
|-----------|---------------|-----|
| `lm_loss` | LM cross-entropy | Teaches experts to predict the next token well |
| `aux_loss` | Load-balance | Prevents expert collapse — keeps all experts active |

Total loss: $\mathcal{L} = \mathcal{L}_{LM} + \lambda \cdot \mathcal{L}_{\text{aux}}$  
where $\lambda$ = `AUX_LOSS_COEFF` (default 0.01).

In [ ]:
# ── Cell 8: Expert Fine-tuning Loop ──

# ── Optimizer — only over trainable (expert) params ──
# Separate 2D weight matrices (decay) from 1D router weights (no decay)
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim >= 2 and "norm" not in name and "bias" not in name:
        decay.append(param)
    else:
        no_decay.append(param)

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8,
    fused=(device.type == "cuda"),
)

n_decay_p    = sum(p.numel() for p in decay)
n_no_decay_p = sum(p.numel() for p in no_decay)
print(f"Optimizer: {n_decay_p:,} decay params, {n_no_decay_p:,} no-decay params")

# ── Mixed precision ──
use_amp = dtype in (torch.float16, torch.bfloat16) and device.type == "cuda"
amp_ctx = torch.amp.autocast(device_type="cuda", dtype=dtype) if use_amp else nullcontext()
scaler  = torch.amp.GradScaler("cuda", enabled=(use_amp and dtype == torch.float16))

# ── Gradient checkpointing ──
# Recomputes layer activations during backward instead of storing all 32 layers'
# worth of intermediate tensors simultaneously.  Saves ~30–40 % activation memory
# (critical for seq_len=4096, batch=16 on this model) at the cost of one extra
# forward pass per step.
model.enable_gradient_checkpointing()

# ── Clear fragmented CUDA allocations before the loop ──
gc.collect()
torch.cuda.empty_cache()

# ── Training state ──
model.train()
train_iter      = iter(train_loader)
lm_losses       = []
aux_losses      = []
val_losses      = []
best_val_loss   = float("inf")

os.makedirs(EXPERT_CKPT_DIR, exist_ok=True)

print(f"\n{'='*60}")
print(f"Expert fine-tuning: {N_STEPS} steps")
print(f"  MoE layers  : {[i for i, _ in moe_layers]}")
print(f"  n_experts   : {N_EXPERTS}, top_k: {TOP_K}")
print(f"  LR          : {PEAK_LR} → {MIN_LR} (cosine)")
print(f"  Aux coeff   : {AUX_LOSS_COEFF}")
print(f"  Grad ckpt   : enabled")
print(f"{'='*60}\n")

t_start = time.time()

for step in range(N_STEPS):
    # ── LR schedule ──
    lr = get_lr(step, WARMUP_STEPS, N_STEPS, PEAK_LR, MIN_LR)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    # ── Gradient accumulation ──
    optimizer.zero_grad(set_to_none=True)
    step_lm_loss  = 0.0
    step_aux_loss = 0.0

    for micro in range(GRAD_ACCUM):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)

        with amp_ctx:
            out = model(x, targets=y)
            lm_loss = out["loss"] / GRAD_ACCUM

            # Collect auxiliary loss from all MoE layers in this forward pass
            moe_aux = torch.tensor(0.0, device=device)
            for layer in model.layers:
                if isinstance(layer.ffn, SparseMoE):
                    moe_aux = moe_aux + layer.ffn.aux_loss
            moe_aux = moe_aux / GRAD_ACCUM

            total_loss = lm_loss + AUX_LOSS_COEFF * moe_aux

        scaler.scale(total_loss).backward()
        step_lm_loss  += lm_loss.item()
        step_aux_loss += moe_aux.item()

    # ── Optimizer step ──
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    scaler.step(optimizer)
    scaler.update()

    lm_losses.append(step_lm_loss)
    aux_losses.append(step_aux_loss)

    # ── Logging ──
    if step % 50 == 0 or step == N_STEPS - 1:
        elapsed = time.time() - t_start
        steps_per_sec = (step + 1) / elapsed if elapsed > 0 else 0
        print(f"step {step:>5d}/{N_STEPS} | lm_loss={step_lm_loss:.4f} | "
              f"aux={step_aux_loss:.4f} | lr={lr:.2e} | "
              f"{steps_per_sec:.1f} steps/s")

    # ── Validation ──
    if step % 200 == 0 or step == N_STEPS - 1:
        model.eval()
        val_loss_acc = 0.0
        n_val_batches = min(20, len(val_loader))
        val_iter_tmp = iter(val_loader)
        with torch.no_grad():
            for _ in range(n_val_batches):
                try:
                    vx, vy = next(val_iter_tmp)
                except StopIteration:
                    break
                vx, vy = vx.to(device), vy.to(device)
                with amp_ctx:
                    vout = model(vx, targets=vy)
                val_loss_acc += vout["loss"].item()
        val_loss = val_loss_acc / n_val_batches
        val_losses.append((step, val_loss))
        print(f"  → val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Save best checkpoint (state_dict only — no optimizer needed)
            torch.save({"step": step,
                        "model_state_dict": model.state_dict(),
                        "val_loss": val_loss,
                        "model_config": vars(model_config),
                        "n_experts": N_EXPERTS,
                        "top_k": TOP_K,
                        "expert_layers": target_layer_indices},
                       os.path.join(EXPERT_CKPT_DIR, "expert_best.pt"))
            print(f"  ✓ Best checkpoint saved (val_loss={val_loss:.4f})")

        model.train()

print(f"\nFine-tuning complete! Best val loss: {best_val_loss:.4f}")

## 9. Training Curves

In [ ]:
# ── Cell 9: Plot Training Curves ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LM loss
axes[0].plot(lm_losses)
axes[0].set_title("LM Loss (train)")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].grid(True, alpha=0.3)

# Aux loss
axes[1].plot(aux_losses, color="orange")
axes[1].set_title("Aux Load-Balance Loss (train)")
axes[1].set_xlabel("step")
axes[1].set_ylabel("loss")
axes[1].grid(True, alpha=0.3)

# Val loss
if val_losses:
    val_steps, val_vals = zip(*val_losses)
    axes[2].plot(val_steps, val_vals, marker="o", color="green")
    axes[2].set_title("Validation LM Loss")
    axes[2].set_xlabel("step")
    axes[2].set_ylabel("loss")
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Sample Output — After Expert Fine-tuning

Same prompts as step 3b — the expert model has now fine-tuned on domain data.  
With enough steps you should see more coherent, topic-consistent continuations compared to the baseline.

In [ ]:
# ── Cell 10: Sample Output — After Expert Training ──
# Same prompts as the pre-training cell — compare outputs to see what the experts learned.

model.eval()

print("=" * 60)
print("Expert model — AFTER fine-tuning")
print("=" * 60)
for prompt in SAMPLE_PROMPTS:
    out = generate_text(model, tokenizer, prompt,
                        GEN_MAX_TOKENS, GEN_TEMPERATURE, GEN_TOP_K, GEN_TOP_P, device)
    print(f"\nPrompt : {prompt}")
    print(f"Output : {out[:250]}")
print("=" * 60)

## 11. Save Expert Checkpoint & Upload to Google Drive

Saves the **full model state** (base + expert weights) plus MoE metadata to `EXPERT_CKPT_DIR`.  
The checkpoint is self-contained — you can reconstruct the expert-augmented model from it without the original base checkpoint.

If `ENABLE_GDRIVE = True`, both the `expert_latest.pt` and `expert_best.pt` (lowest val loss) files are uploaded to `EXPERT_GDRIVE_FOLDER_ID` automatically.

In [ ]:
# ── Cell 11: Save Expert Checkpoint & Upload to Google Drive ──

expert_ckpt = {
    "step": N_STEPS,
    "model_state_dict": model.state_dict(),
    "model_config": vars(model_config),
    # MoE metadata — needed to reconstruct the augmented model on reload
    "n_experts": N_EXPERTS,
    "top_k": TOP_K,
    "expert_layers": target_layer_indices,
    "warm_start": WARM_START_EXPERTS,
    "base_checkpoint": os.path.join(CKPT_DIR, CKPT_FILE),
    "lm_losses": lm_losses,
    "val_losses": val_losses,
}

save_path = os.path.join(EXPERT_CKPT_DIR, "expert_latest.pt")
torch.save(expert_ckpt, save_path)
size_mb = os.path.getsize(save_path) / 1e6
print(f"Expert checkpoint saved : {save_path}  ({size_mb:.1f} MB)")
print(f"  Keys: {list(expert_ckpt.keys())}")

# ── Upload to Google Drive ──
if ENABLE_GDRIVE and EXPERT_GDRIVE_FOLDER_ID:
    try:
        from src.gdrive import upload_to_gdrive, list_remote_checkpoints

        # Upload expert_latest.pt
        file_id = upload_to_gdrive(save_path, EXPERT_GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
        print(f"\nUploaded expert_latest.pt → Google Drive (id: {file_id})")

        # Upload expert_best.pt if it exists
        best_path = os.path.join(EXPERT_CKPT_DIR, "expert_best.pt")
        if os.path.exists(best_path):
            best_id = upload_to_gdrive(best_path, EXPERT_GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
            print(f"Uploaded expert_best.pt  → Google Drive (id: {best_id})")

        # List what's now on Drive
        remote = list_remote_checkpoints(EXPERT_GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
        print(f"\nCheckpoints on Google Drive ({len(remote)}):")
        for f in remote:
            print(f"  {f['name']:30s}  {f.get('createdTime', '')}")
    except Exception as e:
        print(f"Google Drive upload failed: {e}")
else:
    print("\nGoogle Drive upload skipped (ENABLE_GDRIVE=False or EXPERT_GDRIVE_FOLDER_ID not set)")

## 12. Reload — Reconstruct Expert Model from Checkpoint

Demonstrates the full reload cycle: build base architecture → graft MoE layers → load state dict.  
This is what you would run in a downstream inference or evaluation notebook.

In [ ]:
# ── Cell 11: Reload Expert Checkpoint ──

def load_expert_model(checkpoint_path: str, device: torch.device) -> TransformerLM:
    """Reconstruct an expert-augmented TransformerLM from a saved checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Rebuild model config from saved dict
    cfg = ModelConfig(**ckpt["model_config"])
    loaded_model = TransformerLM(cfg)

    # Re-graft MoE layers with saved hyperparameters
    for idx in ckpt["expert_layers"]:
        loaded_model.layers[idx].ffn = SparseMoE(
            dim=cfg.dim,
            ffn_dim=cfg.ffn_dim,
            n_experts=ckpt["n_experts"],
            top_k=ckpt["top_k"],
            dropout=cfg.dropout,
        )

    # Now the architecture matches the saved state dict — load weights
    loaded_model.load_state_dict(ckpt["model_state_dict"])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    return loaded_model


# Test reload
reloaded = load_expert_model(save_path, device)
print(f"Expert model reloaded successfully.")
print(reloaded.summary())

## 12b. Expand Experts — Add One Expert at a Time

Use `expand_moe_experts()` to grow the expert pool incrementally across training rounds.

**Incremental workflow:**

| Round | Start from | Action | What trains |
|-------|-----------|--------|-------------|
| 1 | Base checkpoint | Run cells 5–11 with `N_EXPERTS=1, TOP_K=1` | Expert 0 |
| 2 | `expert_latest.pt` | Run this cell → re-run cells 6–11 | Expert 1 only; Expert 0 frozen |
| 3 | `expert_latest.pt` | Run this cell → re-run cells 6–11 | Expert 2 only; Experts 0–1 frozen |
| … | … | … | … |

**How the router grows:**
The router weight matrix expands from `(dim, n)` → `(dim, n+1)`.
Old columns are copied exactly — existing routing decisions are preserved.
The new column is initialised near-zero so the new expert must **earn** its routing probability through training rather than stealing load immediately.

> After each expansion: re-run **cell 6** (shows updated parameter budget) then **cells 7–11** (data → train → save).

In [ ]:
# ── Cell 12b: Expand MoE Experts (incremental growth) ──

def expand_moe_experts(
    model: TransformerLM,
    n_new: int = 1,
    warm_start: bool = True,
    device: torch.device = device,
) -> int:
    """
    Add `n_new` experts to every SparseMoE layer in `model`.

    - Old expert weights  → frozen immediately.
    - New ExpertFFN(s)    → trainable (warm-started from expert[0] if warm_start=True).
    - Router              → expanded in-place; old columns preserved, new column
                            near-zero init so the router starts neutral toward the
                            new expert.

    Returns the updated total number of experts per MoE layer.
    """
    new_n = None

    for layer in model.layers:
        if not isinstance(layer.ffn, SparseMoE):
            continue

        moe: SparseMoE = layer.ffn
        old_n   = moe.n_experts
        new_n   = old_n + n_new
        dim     = moe.router.in_features
        ffn_dim = moe.experts[0].w_gate.out_features
        dropout = (moe.experts[0].dropout.p
                   if isinstance(moe.experts[0].dropout, nn.Dropout) else 0.0)

        # 1. Freeze all existing experts
        for expert in moe.experts:
            for p in expert.parameters():
                p.requires_grad_(False)

        # 2. Append new trainable expert(s)
        for _ in range(n_new):
            new_expert = ExpertFFN(dim, ffn_dim, dropout).to(device)
            if warm_start:
                with torch.no_grad():
                    new_expert.w_gate.weight.copy_(moe.experts[0].w_gate.weight)
                    new_expert.w_up.weight.copy_(moe.experts[0].w_up.weight)
                    new_expert.w_down.weight.copy_(moe.experts[0].w_down.weight)
            moe.experts.append(new_expert)

        # 3. Expand router: (old_n, dim) → (new_n, dim)
        old_w      = moe.router.weight.data          # (old_n, dim)
        new_router = nn.Linear(dim, new_n, bias=False).to(device)
        with torch.no_grad():
            new_router.weight[:old_n] = old_w        # preserve existing routing
            nn.init.normal_(new_router.weight[old_n:], mean=0.0, std=1e-3)
        moe.router    = new_router
        moe.n_experts = new_n

    if new_n is None:
        print("No SparseMoE layers found — nothing to expand.")
        return 0

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Experts expanded: {new_n - n_new} → {new_n} per MoE layer")
    print(f"  Old experts : FROZEN")
    print(f"  New experts : trainable  (warm_start={warm_start})")
    print(f"  Router      : trainable  (old weights preserved, new column near-zero)")
    print(f"\nParameter budget after expansion:")
    print(f"  Total      : {total:>12,}  ({total/1e6:.1f}M)")
    print(f"  Trainable  : {trainable:>12,}  ({trainable/1e6:.1f}M)  "
          f"({100*trainable/total:.1f}%)")
    return new_n


# ── Run expansion ──
# Reload the last saved checkpoint, grow by 1 expert, prepare for the next round.
# After running this cell: re-run cell 6 (freeze audit) then cells 7–11 (train + save).

model     = load_expert_model(save_path, device)
N_EXPERTS = expand_moe_experts(model, n_new=1, warm_start=WARM_START_EXPERTS)
TOP_K     = min(TOP_K + 1, N_EXPERTS)  # optionally allow one more active expert per token

# Rebuild moe_layers list used by the training loop and load-distribution cell
moe_layers = [(i, layer.ffn) for i, layer in enumerate(model.layers)
              if isinstance(layer.ffn, SparseMoE)]

print(f"\nReady for round {N_EXPERTS} training.")
print(f"  N_EXPERTS={N_EXPERTS}, TOP_K={TOP_K}")
print(f"  MoE layers: {[i for i, _ in moe_layers]}")
print(f"\nNext steps: re-run cell 6 (parameter audit) then cells 7–11 (train + save).")

## 13. Expert Load Distribution

Visualize how evenly tokens are distributed across experts after training.  
A uniform distribution means the load-balance loss did its job.  
A skewed distribution means some experts dominate — try increasing `AUX_LOSS_COEFF`.

In [ ]:
# ── Cell 12: Expert Load Distribution ──

def measure_expert_load(model: TransformerLM, loader, n_batches: int = 30, device=device):
    """Run n_batches through the model and tally token routing counts."""
    model.eval()

    # Register forward hooks to capture router decisions
    load_counts = {}  # layer_idx -> tensor(n_experts)
    hooks = []

    def make_hook(layer_idx, n_experts):
        def hook(module: SparseMoE, inp, out):
            # After forward, router already ran — recompute for logging only
            with torch.no_grad():
                x = inp[0].view(-1, inp[0].shape[-1])
                probs = F.softmax(module.router(x), dim=-1)
                _, topk_idx = torch.topk(probs, module.top_k, dim=-1)  # (N, top_k)
                counts = torch.zeros(n_experts, device=x.device)
                counts.scatter_add_(0, topk_idx.view(-1),
                                    torch.ones(topk_idx.numel(), device=x.device))
                if layer_idx not in load_counts:
                    load_counts[layer_idx] = torch.zeros(n_experts, device=x.device)
                load_counts[layer_idx] += counts
        return hook

    for i, layer in enumerate(model.layers):
        if isinstance(layer.ffn, SparseMoE):
            h = layer.ffn.register_forward_hook(make_hook(i, N_EXPERTS))
            hooks.append(h)

    it = iter(loader)
    with torch.no_grad():
        for _ in range(n_batches):
            try:
                x, _ = next(it)
            except StopIteration:
                break
            model(x.to(device))

    for h in hooks:
        h.remove()

    return load_counts


load_counts = measure_expert_load(reloaded, val_loader, n_batches=20, device=device)

n_moe = len(load_counts)
if n_moe > 0:
    fig, axes = plt.subplots(1, n_moe, figsize=(4 * n_moe, 4), squeeze=False)
    for col, (layer_idx, counts) in enumerate(sorted(load_counts.items())):
        fracs = (counts / counts.sum()).cpu().numpy()
        axes[0][col].bar(range(N_EXPERTS), fracs, color="steelblue")
        axes[0][col].axhline(1 / N_EXPERTS, color="red", linestyle="--", label="uniform")
        axes[0][col].set_title(f"Layer {layer_idx} expert load")
        axes[0][col].set_xlabel("Expert index")
        axes[0][col].set_ylabel("Fraction of tokens")
        axes[0][col].legend()
        axes[0][col].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No MoE layers found for load measurement.")